In [ ]:
!pip install -q tensorflow kagglehub scikit-learn

In [ ]:
# Core deep learning library
import tensorflow as tf

# Pre-trained CNN model
from tensorflow.keras.applications import VGG16

# Model and layers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout

# Image preprocessing and augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Training utilities
from tensorflow.keras.callbacks import EarlyStopping

# Dataset download
import kagglehub
import os

# Evaluation
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Plotting
import matplotlib.pyplot as plt

In [ ]:
# Check whether GPU is available
print(tf.config.list_physical_devices('GPU'))

In [ ]:
# Download Kaggle Chest X-ray Pneumonia dataset
dataset_path = kagglehub.dataset_download(
    "paultimothymooney/chest-xray-pneumonia"
)

print("Dataset path:", dataset_path)


Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Dataset path: /kaggle/input/chest-xray-pneumonia


In [ ]:
# Define train, validation, and test directories
train_dir = os.path.join(dataset_path, "chest_xray/train")
val_dir   = os.path.join(dataset_path, "chest_xray/val")
test_dir  = os.path.join(dataset_path, "chest_xray/test")


In [ ]:
# Image size required by VGG16
IMG_SIZE = (224, 224)

# Number of images processed together
BATCH_SIZE = 32

# Training data generator WITH validation split
train_datagen = ImageDataGenerator(
    rescale=1./255,          # Normalize pixel values
    rotation_range=15,       # Random rotation
    zoom_range=0.1,          # Random zoom
    horizontal_flip=True,    # Random horizontal flip
    validation_split=0.2     # 20% of training data used for validation
)

# Test data generator (NO augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
# -------------------------
# Load training data (80%)
# -------------------------
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training"   # <-- important
)

# -------------------------
# Load validation data (20%)
# -------------------------
val_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation"  # <-- important
)

# -------------------------
# Load test data
# -------------------------
test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

Found 4173 images belonging to 2 classes.
Found 1043 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [ ]:
# Load VGG16 trained on ImageNet
# include_top=False removes the original ImageNet classifier
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)


In [ ]:
# Freeze all convolutional layers
# The model acts as a fixed feature extractor
base_model.trainable = False


In [ ]:
# Flatten convolutional feature maps into a 1D vector
x = Flatten()(base_model.output)

# Fully connected layer for task-specific learning
x = Dense(256, activation="relu")(x)

# Dropout to reduce overfitting
x = Dropout(0.5)(x)

# Output layer for binary classification
output = Dense(1, activation="sigmoid")(x)

# Combine base model and classifier into one model
model = Model(inputs=base_model.input, outputs=output)

In [ ]:
# Compile model
# Adam optimizer + binary cross-entropy for binary classification
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
history_feature = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
106/131 ━━━━━━━━━━━━━━━━━━━━ 9:20 22s/step - accuracy: 0.8044 - loss: 1.1667

In [ ]:
# Enable training of the base model
base_model.trainable = True

# Freeze early layers and fine-tune only deeper layers
for layer in base_model.layers[:-4]:
    layer.trainable = False


In [ ]:
# Smaller learning rate prevents destroying pre-trained weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_finetune = model.fit(
    train_data,
    validation_data=val_data,
    epochs=30,
    callbacks=[early_stop]
)

In [ ]:
# Plot training and validation accuracy
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history_finetune.history['accuracy'], label='Training Accuracy')
plt.plot(history_finetune.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Epochs')
plt.legend()

# Plot training and validation loss
plt.subplot(1,2,2)
plt.plot(history_finetune.history['loss'], label='Training Loss')
plt.plot(history_finetune.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss vs Epochs')
plt.legend()

plt.show()


In [ ]:
test_loss, test_acc = model.evaluate(test_data)
print("Test Accuracy:", test_acc)


In [ ]:
preds = model.predict(test_data)
preds = (preds > 0.5).astype(int)

print(confusion_matrix(test_data.classes, preds))
print(classification_report(test_data.classes, preds))
